#### Decision API 2차 배치 검수

`ner_human_review.review_method = 'LLM검수'` 대상만 PostgreSQL에서 스트리밍 추출하고, 언론사별로 태그 오류와 태깅 병합 필요 여부를 판단한다.

- API 실행은 3번 셀에서 `CURRENT_PRESS`로 지정한 언론사 1곳씩 진행한다.
- 안전 기준 초과 또는 10건 테스트 실패 시 전체 실행은 자동 중단한다.
- 완료된 식별자는 언론사별 `results.jsonl`에서 읽어 재호출하지 않는다.

In [2]:
import os
import sys
from datetime import datetime
from itertools import islice
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path(r'C:\work\ner')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import get_postgre_connect
from src.ner_review.decision.decision import DECISION_MODEL
from src.ner_review.decision.batch import (
    DEFAULT_BATCH_SIZE,
    DEFAULT_MAX_WORKERS,
    DEFAULT_MAX_RETRIES,
    DEFAULT_REQUEST_TIMEOUT_SECONDS,
    build_decision_input,
    estimate_usage,
    run_press_list,
    stream_review_targets,
)

DATA_VERSION = None # os.getenv('DATA_VERSION') or None
MODEL_OVERRIDE = None  # Decisions API 공식 지원 모델 외 값은 사용하지 않음
MODEL = MODEL_OVERRIDE or DECISION_MODEL
if MODEL != 'gpt-6-luna':
    raise ValueError(f'Decisions API 지원 모델은 gpt-6-luna입니다: {MODEL}')
RUN_API = True  # 3번 셀에서 CURRENT_PRESS 언론사 1곳을 실행
PRESS_FILTER = None  # 예: ['강원일보']
OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / 'decision_2' / f'run_{datetime.now():%Y%m%d_%H%M%S}'

# API 제한과 비용에 맞춰 조정하는 설정
BATCH_SIZE = DEFAULT_BATCH_SIZE
MAX_WORKERS = DEFAULT_MAX_WORKERS
MAX_RETRIES = DEFAULT_MAX_RETRIES
BACKOFF_SECONDS = 2.0
MAX_BACKOFF_SECONDS = 60.0
REQUEST_TIMEOUT_SECONDS = DEFAULT_REQUEST_TIMEOUT_SECONDS
MAX_INPUT_CHARS = 120_000
SAMPLE_SIZE = 100
CHARS_PER_TOKEN = 4.0
ESTIMATED_OUTPUT_TOKENS_PER_REQUEST = 300
INPUT_PRICE_PER_1M = None   # 모델 가격표 확인 후 숫자로 입력
OUTPUT_PRICE_PER_1M = None  # 모델 가격표 확인 후 숫자로 입력

print('output_root:', OUTPUT_ROOT)
print('model:', MODEL, '| run_api:', RUN_API)

output_root: C:\work\ner\outputs\decision_2\run_20261120_014739
model: gpt-6-luna | run_api: True


##### 1. 대상 언론사와 소규모 입력 확인

In [3]:
def list_press_names(data_version=None):
    where = "WHERE review_method = 'LLM검수'"
    params = []
    if data_version:
        where += ' AND data_version = %s'
        params.append(data_version)
    with get_postgre_connect() as connection:
        rows = connection.execute(
            f'SELECT DISTINCT press FROM ner_human_review {where} ORDER BY press',
            params,
        ).fetchall()
    return [row[0] for row in rows]

press_list = list_press_names(DATA_VERSION)
if PRESS_FILTER:
    press_list = [press for press in press_list if press in PRESS_FILTER]
print('presses:', press_list)

preview_rows = []
for press in press_list[:3]:
    target = next(stream_review_targets(data_version=DATA_VERSION, press=press), None)
    if target:
        preview_rows.append({
            'press': press,
            'data_version': target['data_version'],
            'doc_id': target['doc_id'],
            'sentence_id': target['sentence_id'],
            'original_form': target['original_form'],
            'entity_count': len(target['entities']),
            'active_rule_count': len(target['sentence_rule_evidence']),
        })
display(pd.DataFrame(preview_rows))

presses: ['강원일보', '경기일보', '경향신문', '노컷뉴스', '뉴스1', '뉴스핌', '뉴시스', '디지털타임스', '마이데일리', '부산일보', '서울신문', '세계일보', '스포츠경향', '스포츠투데이', '아시아투데이', '이데일리', '이투데이', '일간투데이', '충청일보']


,press,data_version,doc_id,sentence_id,original_form,entity_count,active_rule_count
0,강원일보,20261117_192958_강원일보,NLRW2600000001.10006,NLRW2600000001.10006.6.2,<2022년: DT> <20대 대선: EV>의 <강원: LC>지역 <1일차: DT>...,1,1
1,경기일보,20261117_193354_경기일보,NLRW2600000002.10002,NLRW2600000002.10002.1.1,"<박남춘: PS> “<이재명: PS> 원팀, 정권교체”…<1년: DT>만 SNS 글...",2,2
2,경향신문,20261117_193858_경향신문,NWRW2600000001.10004,NWRW2600000001.10004.5.1,앞서 <법사위: OG>는 <윤석열: PS> 대통령의 구속취소 결정에 대한 <이날: ...,1,1


##### 2. 언론사별 요청 수·토큰·비용 추정

전체 입력은 메모리에 모으지 않고, 언론사별 generator를 한 번 순회한다. 토큰은 문자 수 기반 보수적 추정이며 실제 사용량은 결과 JSONL의 `api.usage`를 확인한다.

In [3]:
usage_rows = []
for press in press_list:
    target_iter = stream_review_targets(data_version=DATA_VERSION, press=press)
    sample_targets = list(islice(target_iter, SAMPLE_SIZE))
    usage = estimate_usage(sample_targets, sample_size=SAMPLE_SIZE, chars_per_token=CHARS_PER_TOKEN)
    remaining_count = sum(1 for _ in target_iter)
    estimated_request_count = len(sample_targets) + remaining_count
    estimated_input_tokens = usage['estimated_input_tokens']
    estimated_input_tokens_total = estimated_request_count * estimated_input_tokens
    estimated_output_tokens_total = estimated_request_count * ESTIMATED_OUTPUT_TOKENS_PER_REQUEST
    estimated_cost = None
    if INPUT_PRICE_PER_1M is not None and OUTPUT_PRICE_PER_1M is not None:
        estimated_cost = (estimated_input_tokens_total / 1_000_000 * INPUT_PRICE_PER_1M) + (estimated_output_tokens_total / 1_000_000 * OUTPUT_PRICE_PER_1M)
    usage_rows.append({
        'press': press,
        'sample_size': usage['sample_size'],
        'estimated_request_count': estimated_request_count,
        'avg_input_chars': usage['avg_input_chars'],
        'max_input_chars': usage.get('max_input_chars', 0),
        'estimated_input_tokens_per_request': estimated_input_tokens,
        'estimated_input_tokens_total': estimated_input_tokens_total,
        'estimated_output_tokens_total': estimated_output_tokens_total,
        'estimated_cost': estimated_cost,
    })
usage_frame = pd.DataFrame(usage_rows)
display(usage_frame)
print('주의: 정확한 요청 수/사용 토큰은 각 press/summary.json에 누적 기록됩니다.')

,press,sample_size,estimated_request_count,avg_input_chars,max_input_chars,estimated_input_tokens_per_request,estimated_input_tokens_total,estimated_output_tokens_total,estimated_cost
0,강원일보,100,2177,504.8,765,126,274302,653100,None
1,경기일보,100,2706,537.7,863,134,362604,811800,None
2,경향신문,100,6372,525.1,797,131,834732,1911600,None
3,노컷뉴스,100,9076,551.5,1183,138,1252488,2722800,None
4,뉴스1,100,28855,545.4,840,136,3924280,8656500,None
5,뉴스핌,100,14403,552.1,1024,138,1987614,4320900,None
6,뉴시스,100,22106,534.3,1207,134,2962204,6631800,None
7,디지털타임스,100,5404,540.0,751,135,729540,1621200,None
8,마이데일리,100,7706,519.0,745,130,1001780,2311800,None
9,부산일보,100,6347,543.5,952,136,863192,1904100,None


주의: 정확한 요청 수/사용 토큰은 각 press/summary.json에 누적 기록됩니다.


##### 3. 언론사별 순차 배치 실행

`MAX_WORKERS`만 한 언론사 내부에서 병렬 처리하고, 언론사 자체는 순차 처리한다. 배치가 끝날 때마다 `results.jsonl`, `failures.jsonl`, `summary.json`, `execution.log.jsonl`을 저장한다.

In [4]:
CURRENT_PRESS = '경기일보'  # 실행할 언론사를 1곳씩 지정

def get_press_data_version(press):
    with get_postgre_connect() as connection:
        rows = connection.execute(
            "SELECT DISTINCT data_version FROM ner_human_review "
            "WHERE review_method = 'LLM검수' AND press = %s ORDER BY data_version DESC",
            [press],
        ).fetchall()
    data_versions = [row[0] for row in rows]
    if not data_versions:
        raise ValueError(f'ner_human_review에 데이터가 없습니다: {press}')
    if DATA_VERSION and DATA_VERSION not in data_versions:
        raise ValueError(f'{press}에 DATA_VERSION이 없습니다: {DATA_VERSION}')
    if DATA_VERSION:
        return DATA_VERSION
    if len(data_versions) > 1:
        raise ValueError(f'{press}에 여러 data_version이 있습니다: {data_versions}')
    return data_versions[0]

if not CURRENT_PRESS:
    raise ValueError('CURRENT_PRESS에 실행할 언론사를 지정하세요.')
if CURRENT_PRESS not in press_list:
    raise ValueError(f'등록되지 않은 언론사입니다: {CURRENT_PRESS}')
PRESS_DATA_VERSION = get_press_data_version(CURRENT_PRESS)
print(f'실행 언론사: {CURRENT_PRESS} | data_version: {PRESS_DATA_VERSION}')

if RUN_API:
    summaries = run_press_list(
        [CURRENT_PRESS],
        OUTPUT_ROOT,
        data_version=PRESS_DATA_VERSION,
        model=MODEL,
        batch_size=BATCH_SIZE,
        max_workers=MAX_WORKERS,
        max_retries=MAX_RETRIES,
        backoff_seconds=BACKOFF_SECONDS,
        max_backoff_seconds=MAX_BACKOFF_SECONDS,
        request_timeout_seconds=REQUEST_TIMEOUT_SECONDS,
        max_input_chars=MAX_INPUT_CHARS,
        progress=lambda event: print(event),
    )
    display(pd.DataFrame(summaries))
else:
    print('RUN_API=False: 실제 Decision API 호출을 실행하지 않았습니다.')

실행 언론사: 경기일보 | data_version: 20261117_193354_경기일보
{'event': 'completed', 'press': '경기일보', 'doc_id': 'NLRW2600000002.1003', 'sentence_id': 'NLRW2600000002.1003.5.2', 'attempts': 1}
{'event': 'completed', 'press': '경기일보', 'doc_id': 'NLRW2600000002.10040', 'sentence_id': 'NLRW2600000002.10040.10.1', 'attempts': 1}
{'event': 'completed', 'press': '경기일보', 'doc_id': 'NLRW2600000002.10002', 'sentence_id': 'NLRW2600000002.10002.1.1', 'attempts': 1}
{'event': 'completed', 'press': '경기일보', 'doc_id': 'NLRW2600000002.10045', 'sentence_id': 'NLRW2600000002.10045.1.1', 'attempts': 1}
{'event': 'completed', 'press': '경기일보', 'doc_id': 'NLRW2600000002.10090', 'sentence_id': 'NLRW2600000002.10090.9.1', 'attempts': 1}
{'event': 'completed', 'press': '경기일보', 'doc_id': 'NLRW2600000002.10112', 'sentence_id': 'NLRW2600000002.10112.3.1', 'attempts': 1}
{'event': 'completed', 'press': '경기일보', 'doc_id': 'NLRW2600000002.1007', 'sentence_id': 'NLRW2600000002.1007.3.2', 'attempts': 1}
{'event': 'completed', 'press

,schema_version,run_id,press,data_version,model,prompt_version,started_at,status,batch_size,max_workers,...,failed,skipped,pending,input_tokens,output_tokens,request_count,retried_requests,retry_targets,unprocessed,finished_at
0,ner-decision-v2,run_20261120_014739,경기일보,20261117_193354_경기일보,gpt-6-luna,decision-v2,2026-11-19T16:48:08.148456+00:00,completed,20,4,...,0,0,0,22490855,0,2706,0,0,0,2026-11-19T16:51:55.069643+00:00


##### 4. 실행 현황과 결과 확인

In [5]:
def load_summaries(output_root):
    paths = sorted(Path(output_root).glob('press/*/summary.json'))
    rows = []
    for path in paths:
        rows.append(__import__('json').loads(path.read_text(encoding='utf-8')).copy())
        rows[-1]['summary_path'] = str(path)
    return pd.DataFrame(rows)

summary_frame = load_summaries(OUTPUT_ROOT)
if CURRENT_PRESS:
    summary_frame = summary_frame[summary_frame['press'] == CURRENT_PRESS]
display(summary_frame)

press_dir = Path(OUTPUT_ROOT) / 'press' / CURRENT_PRESS
result_path = press_dir / 'results.jsonl'
failure_path = press_dir / 'failures.jsonl'
if result_path.exists():
    result_frame = pd.read_json(result_path, lines=True)
    display(result_frame)
if failure_path.exists():
    failure_frame = pd.read_json(failure_path, lines=True)
    display(failure_frame)
print('결과 루트:', OUTPUT_ROOT)

,schema_version,run_id,press,data_version,model,prompt_version,started_at,status,batch_size,max_workers,...,skipped,pending,input_tokens,output_tokens,request_count,retried_requests,retry_targets,unprocessed,finished_at,summary_path
0,ner-decision-v2,run_20261120_014739,경기일보,20261117_193354_경기일보,gpt-6-luna,decision-v2,2026-11-19T16:48:08.148456+00:00,completed,20,4,...,0,0,22490855,0,2706,0,0,0,2026-11-19T16:51:55.069643+00:00,C:\work\ner\outputs\decision_2\run_20261120_01...


,schema_version,status,run_id,executed_at,model,prompt_version,data_version,press,doc_id,sentence_id,original_form,tagging_error,tagging_merge,source,api
0,ner-decision-v2,completed,run_20261120_014739,2026-11-19 16:48:09.604858+00:00,gpt-6-luna,decision-v2,20261117_193354_경기일보,경기일보,NLRW2600000002.10002,NLRW2600000002.10002.1.1,"<박남춘: PS> “<이재명: PS> 원팀, 정권교체”…<1년: DT>만 SNS 글...","{'is_error': False, 'error_type': None, 'targe...","{'needed': False, 'merge_entity_ids': [], 'mer...",{'existing_entities': [{'entity_id': 'NLRW2600...,"{'response_id': None, 'attempts': 1, 'input_ch..."
1,ner-decision-v2,completed,run_20261120_014739,2026-11-19 16:48:09.591388+00:00,gpt-6-luna,decision-v2,20261117_193354_경기일보,경기일보,NLRW2600000002.1003,NLRW2600000002.1003.5.2,<권: PS> 대표는 “출범한 지 <5년: DT> 동안 구속영장과 체포영장 한 번 ...,"{'is_error': False, 'error_type': None, 'targe...","{'needed': False, 'merge_entity_ids': [], 'mer...",{'existing_entities': [{'entity_id': 'NLRW2600...,"{'response_id': None, 'attempts': 1, 'input_ch..."
2,ner-decision-v2,completed,run_20261120_014739,2026-11-19 16:48:09.594182+00:00,gpt-6-luna,decision-v2,20261117_193354_경기일보,경기일보,NLRW2600000002.10040,NLRW2600000002.10040.10.1,<시: OG>는 <다음 달: DT>부터 본격적인 입소자 신청 접수를 시작할 예정이다.,"{'is_error': False, 'error_type': None, 'targe...","{'needed': False, 'merge_entity_ids': [], 'mer...",{'existing_entities': [{'entity_id': 'NLRW2600...,"{'response_id': None, 'attempts': 1, 'input_ch..."
3,ner-decision-v2,completed,run_20261120_014739,2026-11-19 16:48:09.642017+00:00,gpt-6-luna,decision-v2,20261117_193354_경기일보,경기일보,NLRW2600000002.10045,NLRW2600000002.10045.1.1,"<경찰: OG>, <대선: REVIEW_EV> 후보 경호 시작…<6·3 대선: EV...","{'is_error': False, 'error_type': None, 'targe...","{'needed': False, 'merge_entity_ids': [], 'mer...",{'existing_entities': [{'entity_id': 'NLRW2600...,"{'response_id': None, 'attempts': 1, 'input_ch..."
4,ner-decision-v2,completed,run_20261120_014739,2026-11-19 16:48:09.889264+00:00,gpt-6-luna,decision-v2,20261117_193354_경기일보,경기일보,NLRW2600000002.1007,NLRW2600000002.1007.3.2,<올해: DT>부터는 횟수 제한 없이 1인당 최대 10만원까지 어학시험 및 국가자격...,"{'is_error': False, 'error_type': None, 'targe...","{'needed': False, 'merge_entity_ids': [], 'mer...",{'existing_entities': [{'entity_id': 'NLRW2600...,"{'response_id': None, 'attempts': 1, 'input_ch..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2701,ner-decision-v2,completed,run_20261120_014739,2026-11-19 16:51:54.761284+00:00,gpt-6-luna,decision-v2,20261117_193354_경기일보,경기일보,NLRW2600000002.9990,NLRW2600000002.9990.8.1,<이: PS> 후보는 <이날: DT> 수락연설에서 “<23년 전: DT> <오늘: ...,"{'is_error': False, 'error_type': None, 'targe...","{'needed': False, 'merge_entity_ids': [], 'mer...",{'existing_entities': [{'entity_id': 'NLRW2600...,"{'response_id': None, 'attempts': 1, 'input_ch..."
2702,ner-decision-v2,completed,run_20261120_014739,2026-11-19 16:51:54.741972+00:00,gpt-6-luna,decision-v2,20261117_193354_경기일보,경기일보,NLRW2600000002.9991,NLRW2600000002.9991.1.1,<이재명: PS> “대통령 제1과제 국민통합 책임 확실하게 완수” [<6·3 대선:...,"{'is_error': False, 'error_type': None, 'targe...","{'needed': False, 'merge_entity_ids': [], 'mer...",{'existing_entities': [{'entity_id': 'NLRW2600...,"{'response_id': None, 'attempts': 1, 'input_ch..."
2703,ner-decision-v2,completed,run_20261120_014739,2026-11-19 16:51:54.760290+00:00,gpt-6-luna,decision-v2,20261117_193354_경기일보,경기일보,NLRW2600000002.9996,NLRW2600000002.9996.1.1,"<이재명: PS> “압도적 지지, 압도적 기대라 생각…무겁게 받아들일 것” [<6·...","{'is_error': False, 'error_type': None, 'targe...","{'needed': False, 'merge_entity_ids': [], 'mer...",{'existing_entities': [{'entity_id': 'NLRW2600...,"{'response_id': None, 'attempts': 1, 'input_ch..."
2704,ner-decision-v2,completed,run_20261120_014739,2026-11-19 16:51:54.997929+00:00,gpt-6-luna,decision-v2,20261117_193354_경기일보,경기일보,NLRW2600000002.9998,NLRW2600000002.9998.1.1,막 내린 <김동연: PS> <경기도지사: OG>의 도전…역부족 속에도 부각된 존재감...,"{'is_error': False, 'error_type': None, 'targe...","{'needed': False, 'merge_entity_ids': [], 

""


결과 루트: C:\work\ner\outputs\decision_2\run_20261120_014739


In [ ]:
# 재개 실행 예시: 같은 OUTPUT_ROOT를 사용하면 completed 식별자를 건너뛴다.
# 실패 건은 failures.jsonl을 확인한 뒤 필요하면 같은 press만 다시 실행한다.
# run_press('강원일보', OUTPUT_ROOT, data_version=DATA_VERSION, model=MODEL,
#           batch_size=BATCH_SIZE, max_workers=MAX_WORKERS, max_retries=MAX_RETRIES)

##### 실행 순서(언론사별 수동 실행)

1. 위 설정 셀과 1·2번 셀을 먼저 실행한다.
2. 3번 셀의 `CURRENT_PRESS`에 실행할 언론사 1곳을 지정하고 3번 셀만 실행한다.
3. 실행이 끝나면 4번 셀을 실행해 해당 언론사의 summary와 결과를 확인한다.
4. 다음 언론사를 실행할 때는 `CURRENT_PRESS`만 바꾼 뒤 3번과 4번 셀을 반복 실행한다.
5. 모든 언론사는 같은 `OUTPUT_ROOT` 아래 언론사별 폴더에 저장된다.

In [4]:
raise RuntimeError('AUTO_RUN 셀은 사용하지 않습니다. 3번 셀에서 CURRENT_PRESS를 지정해 실행하세요.')
import importlib
import src.ner_review.decision.batch as batch_module
importlib.reload(batch_module)
from src.ner_review.decision.batch import count_completed_results, run_press_with_remediation

MAX_SAFE_INPUT_CHARS_PER_REQUEST = MAX_INPUT_CHARS
MAX_SAFE_INPUT_TOKENS_PER_REQUEST = 12000
MAX_SAFE_TOTAL_TOKENS = 2000000
TEST_LIMIT = 10

if not press_list:
    raise RuntimeError('LLM검수 대상 언론사가 없습니다.')

unsafe_rows = usage_frame[
    (usage_frame['max_input_chars'] > MAX_SAFE_INPUT_CHARS_PER_REQUEST)
    | (usage_frame['estimated_input_tokens_per_request'] > MAX_SAFE_INPUT_TOKENS_PER_REQUEST)
].copy()
total_input_tokens = int(usage_frame['estimated_input_tokens_total'].sum())
total_output_tokens = int(usage_frame['estimated_output_tokens_total'].sum())
total_estimated_tokens = total_input_tokens + total_output_tokens
print(f'예상 전체 입력 토큰: {total_input_tokens:,}')
print(f'예상 전체 출력 토큰: {total_output_tokens:,}')
print(f'안전 기준: 요청당 입력 {MAX_SAFE_INPUT_TOKENS_PER_REQUEST:,} tokens, 전체 {MAX_SAFE_TOTAL_TOKENS:,} tokens')

if not unsafe_rows.empty:
    display(unsafe_rows[['press', 'max_input_chars', 'estimated_input_tokens_per_request']])
    raise RuntimeError('입력 크기 또는 요청당 토큰이 안전 기준을 초과하여 API 실행을 중단했습니다.')
if total_estimated_tokens > MAX_SAFE_TOTAL_TOKENS:
    raise RuntimeError('전체 예상 입력+출력 토큰이 안전 기준을 초과하여 API 실행을 중단했습니다.')

test_press = press_list[0]
expected_test_count = min(TEST_LIMIT, int(usage_frame.loc[usage_frame['press'] == test_press, 'estimated_request_count'].iloc[0]))
test_output_root = PROJECT_ROOT / 'outputs' / 'decision_2_test' / f'run_{datetime.now():%Y%m%d_%H%M%S}'
test_summary = run_press_with_remediation(
    test_press, test_output_root, data_version=DATA_VERSION, model=MODEL, limit=TEST_LIMIT,
    batch_size=min(BATCH_SIZE, TEST_LIMIT), max_workers=MAX_WORKERS, max_retries=MAX_RETRIES,
    backoff_seconds=BACKOFF_SECONDS, max_backoff_seconds=MAX_BACKOFF_SECONDS,
    request_timeout_seconds=REQUEST_TIMEOUT_SECONDS, max_input_chars=MAX_INPUT_CHARS,
    progress=lambda event: print(event),
)
test_dir = test_output_root / 'press' / test_press
test_failed = int(test_summary.get('failed', 0))
test_completed = count_completed_results(test_output_root, test_press)
print(f'10건 테스트 완료: completed={test_completed}/{expected_test_count}, failed={test_failed}, remediation={test_summary.get("remediation")}')
if (test_dir / 'results.jsonl').exists():
    result_frame = pd.read_json(test_dir / 'results.jsonl', lines=True)
    result_columns = ['press', 'doc_id', 'sentence_id', 'tagging_error', 'tagging_merge', 'status']
    display(result_frame[[column for column in result_columns if column in result_frame.columns]])
if (test_dir / 'failures.jsonl').exists():
    display(pd.read_json(test_dir / 'failures.jsonl', lines=True))
if test_failed or test_completed < expected_test_count:
    raise RuntimeError('자동 복구 후에도 10건 테스트가 완료되지 않아 전체 실행을 중단했습니다. failures.jsonl을 확인하세요.')

full_summaries = run_press_list(
    press_list, OUTPUT_ROOT, data_version=DATA_VERSION, model=MODEL,
    batch_size=BATCH_SIZE, max_workers=MAX_WORKERS, max_retries=MAX_RETRIES,
    backoff_seconds=BACKOFF_SECONDS, max_backoff_seconds=MAX_BACKOFF_SECONDS,
    request_timeout_seconds=REQUEST_TIMEOUT_SECONDS, max_input_chars=MAX_INPUT_CHARS,
    progress=lambda event: print(event),
)
display(pd.DataFrame(full_summaries))

예상 전체 입력 토큰: 23,913,711
예상 전체 출력 토큰: 53,326,500
안전 기준: 요청당 입력 12,000 tokens, 전체 2,000,000 tokens


RuntimeError: 전체 예상 입력+출력 토큰이 안전 기준을 초과하여 API 실행을 중단했습니다.

In [22]:
# Decision 결과 확인용 테이블
result_paths = sorted((Path(OUTPUT_ROOT) / 'press').glob('*/results.jsonl'))
result_records = []
for result_path in result_paths:
    if result_path.exists():
        result_records.extend(pd.read_json(result_path, lines=True).to_dict('records'))

if result_records:
    decision_table = pd.json_normalize(result_records)
    decision_columns = [
        'data_version', 'press', 'doc_id', 'sentence_id', 'original_form', 'status',
        'tagging_error.is_error', 'tagging_error.error_type', 'tagging_error.confidence',
        'tagging_merge.needed', 'tagging_merge.confidence',
        'api.attempts', 'api.input_chars', 'api.usage.input_tokens',
        'api.usage.output_tokens', 'api.response_id',
    ]
    decision_table = decision_table[[column for column in decision_columns if column in decision_table.columns]]
    decision_table = decision_table.sort_values(['press', 'doc_id', 'sentence_id']).reset_index(drop=True)
    print(f'Decision 결과: {len(decision_table):,}건')
    display(decision_table)
    display(decision_table['status'].value_counts(dropna=False).rename_axis('status').reset_index(name='count'))
else:
    print(f'결과 파일이 없습니다: {OUTPUT_ROOT / "press"}')

Decision 결과: 2,177건


,data_version,press,doc_id,sentence_id,original_form,status,tagging_error.is_error,tagging_error.error_type,tagging_error.confidence,tagging_merge.needed,tagging_merge.confidence,api.attempts,api.input_chars,api.usage.input_tokens,api.usage.output_tokens,api.response_id
0,20261117_192958_강원일보,강원일보,NLRW2600000001.10006,NLRW2600000001.10006.6.2,<2022년: DT> <20대 대선: EV>의 <강원: LC>지역 <1일차: DT>...,completed,True,other_tagging_error,0.10,False,1.00,1,501,8076,0,None
1,20261117_192958_강원일보,강원일보,NLRW2600000001.10008,NLRW2600000001.10008.5.1,<6·3 대선: EV>을 앞두고 <강원도: LC>는 다시금 전국 정치의 중심에 섰다.,completed,False,NaN,0.12,False,1.00,1,462,8051,0,None
2,20261117_192958_강원일보,강원일보,NLRW2600000001.10014,NLRW2600000001.10014.1.1,<오늘: DT>부터 사전투표… <이재명: PS>-<김문수: PS> <30일: DT>...,completed,False,NaN,0.10,False,1.00,1,491,8069,0,None
3,20261117_192958_강원일보,강원일보,NLRW2600000001.10014,NLRW2600000001.10014.6.1,<29일: DT> 사전투표를 시작으로 <닷새간: DT> <6·3대선: EV> 운명을...,completed,True,other_tagging_error,0.02,False,1.00,1,473,8062,0,None
4,20261117_192958_강원일보,강원일보,NLRW2600000001.1002,NLRW2600000001.1002.4.1,"복지바우처는 1인당 20만 원씩 총 935명에게 지원되며 지급된 카드는 도내 문화,...",completed,False,NaN,0.06,False,1.00,1,519,8083,0,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2172,20261117_192958_강원일보,강원일보,NLRW2600000001.9993,NLRW2600000001.9993.2.1,<2015년: DT>부터 해마다 500여 건 서비스 군민 비용 절감 편의 증진,completed,True,other_tagging_error,0.10,False,1.00,1,474,8054,0,None
2173,20261117_192958_강원일보,강원일보,NLRW2600000001.9993,NLRW2600000001.9993.5.1,"<군: OG>은 <2015년: DT>부터 농지에 설치하는 20㎡ 이내의 농막, 33...",completed,False,NaN,0.20,False,1.00,1,609,8139,0,None
2174,20261117_192958_강원일보,강원일보,NLRW2600000001.9993,NLRW2600000001.9993.7.1,실제 <군: OG>에 접수된 민원처리 건수는 <2015년: DT>부터 <지난해: D...,completed,True,other_tagging_error,0.46,True,0.38,1,725,8171,0,None
2175,20261117_192958_강원일보,강원일보,NLRW2600000001.9995,NLRW2600000001.9995.4.1,"<평창군: OG>은 <28, 29일: DT> <이틀간: DT> <평창올림픽플라자: ...",completed,True,other_tagging_error,0.06,False,1.00,1,582,8135,0,None


,status,count
0,completed,2177


In [24]:
raise RuntimeError('다중 data_version 일괄 실행 셀은 사용하지 않습니다. 3번과 4번 셀을 언론사별로 반복 실행하세요.')
import importlib
import src.ner_review.decision.batch as batch_module
importlib.reload(batch_module)
from src.ner_review.decision.batch import count_completed_results, run_press_with_remediation

CURRENT_PRESS = None
MAX_SAFE_INPUT_CHARS_PER_REQUEST = MAX_INPUT_CHARS
MAX_SAFE_INPUT_TOKENS_PER_REQUEST = 12000
MAX_SAFE_TOTAL_TOKENS = 2000000
TEST_LIMIT = 10

def list_target_data_versions(excluded_press):
    with get_postgre_connect() as connection:
        rows = connection.execute(
            "SELECT DISTINCT data_version FROM ner_human_review WHERE review_method = 'LLM검수' AND press <> %s ORDER BY data_version",
            [excluded_press],
        ).fetchall()
    return [row[0] for row in rows]

target_data_versions = list_target_data_versions(CURRENT_PRESS)
if not target_data_versions:
    raise RuntimeError(f'{CURRENT_PRESS} 제외 후 실행할 data_version이 없습니다.')
print('실행 대상 data_version:', target_data_versions)

all_full_summaries = []
for data_version in target_data_versions:
    print(f'\n===== data_version 시작: {data_version} =====')
    version_press_list = list_press_names(data_version)
    if PRESS_FILTER:
        version_press_list = [press for press in version_press_list if press in PRESS_FILTER]
    if not version_press_list:
        print('대상 언론사가 없어 건너뜁니다.')
        continue

    version_usage_rows = []
    for press in version_press_list:
        target_iter = stream_review_targets(data_version=data_version, press=press)
        sample_targets = list(islice(target_iter, SAMPLE_SIZE))
        usage = estimate_usage(sample_targets, sample_size=SAMPLE_SIZE, chars_per_token=CHARS_PER_TOKEN)
        remaining_count = sum(1 for _ in target_iter)
        request_count = len(sample_targets) + remaining_count
        version_usage_rows.append({
            'press': press,
            'estimated_request_count': request_count,
            'max_input_chars': usage.get('max_input_chars', 0),
            'estimated_input_tokens_per_request': usage['estimated_input_tokens'],
            'estimated_input_tokens_total': request_count * usage['estimated_input_tokens'],
            'estimated_output_tokens_total': request_count * ESTIMATED_OUTPUT_TOKENS_PER_REQUEST,
        })
    version_usage_frame = pd.DataFrame(version_usage_rows)
    display(version_usage_frame)
    unsafe_rows = version_usage_frame[
        (version_usage_frame['max_input_chars'] > MAX_SAFE_INPUT_CHARS_PER_REQUEST)
        | (version_usage_frame['estimated_input_tokens_per_request'] > MAX_SAFE_INPUT_TOKENS_PER_REQUEST)
    ]
    total_estimated_tokens = int(version_usage_frame['estimated_input_tokens_total'].sum() + version_usage_frame['estimated_output_tokens_total'].sum())
    if not unsafe_rows.empty or total_estimated_tokens > MAX_SAFE_TOTAL_TOKENS:
        display(unsafe_rows)
        raise RuntimeError(f'{data_version}가 안전 기준을 초과하여 이후 실행을 중단했습니다.')

    test_press = version_press_list[0]
    expected_test_count = min(TEST_LIMIT, int(version_usage_frame.loc[version_usage_frame['press'] == test_press, 'estimated_request_count'].iloc[0]))
    test_output_root = PROJECT_ROOT / 'outputs' / 'decision_2_test' / data_version / f'run_{datetime.now():%Y%m%d_%H%M%S_%f}'
    test_summary = run_press_with_remediation(
        test_press, test_output_root, data_version=data_version, model=MODEL, limit=TEST_LIMIT,
        batch_size=min(BATCH_SIZE, TEST_LIMIT), max_workers=MAX_WORKERS, max_retries=MAX_RETRIES,
        backoff_seconds=BACKOFF_SECONDS, max_backoff_seconds=MAX_BACKOFF_SECONDS,
        request_timeout_seconds=REQUEST_TIMEOUT_SECONDS, max_input_chars=MAX_INPUT_CHARS,
        progress=lambda event: print(event),
    )
    test_completed = count_completed_results(test_output_root, test_press)
    test_failed = int(test_summary.get('failed', 0))
    print(f'테스트 결과: {data_version}, completed={test_completed}/{expected_test_count}, failed={test_failed}')
    if test_failed or test_completed < expected_test_count:
        raise RuntimeError(f'{data_version}의 10건 테스트 실패로 전체 실행을 중단했습니다.')

    output_root = PROJECT_ROOT / 'outputs' / 'decision_2' / data_version / f'run_{datetime.now():%Y%m%d_%H%M%S_%f}'
    full_summaries = run_press_list(
        version_press_list, output_root, data_version=data_version, model=MODEL,
        batch_size=BATCH_SIZE, max_workers=MAX_WORKERS, max_retries=MAX_RETRIES,
        backoff_seconds=BACKOFF_SECONDS, max_backoff_seconds=MAX_BACKOFF_SECONDS,
        request_timeout_seconds=REQUEST_TIMEOUT_SECONDS, max_input_chars=MAX_INPUT_CHARS,
        progress=lambda event: print(event),
    )
    all_full_summaries.extend(full_summaries)
    display(pd.DataFrame(full_summaries))
    print(f'===== data_version 완료: {data_version} =====')

display(pd.DataFrame(all_full_summaries))

RuntimeError: None 제외 후 실행할 data_version이 없습니다.